# 04 — Model Training
CS2 Match Outcome Predictor & Seeding Engine

Trains and compares:
1. Logistic Regression (interpretable baseline classifier)
2. XGBoost (main model)

Uses a temporal (time-based) train/test split, MLflow experiment
tracking, and compares both against the Elo-only baseline from
03_baseline_elo.ipynb.

In [1]:
import os

def find_repo_root(marker='requirements.txt'):
    path = os.getcwd()
    while True:
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root (looking for '{marker}')")
        path = parent

repo_root = find_repo_root()
os.chdir(repo_root)

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\User\Desktop\CS model\cs2-seeding-capstone


In [2]:
try:
    import mlflow
    import xgboost
except ImportError:
    %pip install mlflow xgboost -q
    import mlflow
    import xgboost

import logging
import time
import pandas as pd
import numpy as np
import mlflow

try:
    import google.colab
    logging.Formatter.converter = lambda *args: time.localtime(time.time() + 5*3600)
except ImportError:
    pass

logging.basicConfig(
    filename='data/04_model_training.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    filemode='a',
    force=True
)

logging.info("=" * 60)
logging.info("NEW RUN STARTED")
logging.info("=" * 60)

In [3]:
df = pd.read_csv('data/matches_with_elo.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Shape:", df.shape)
logging.info(f"Loaded matches_with_elo.csv, shape: {df.shape}")

Shape: (6989, 59)


## 1. Feature selection

Only verified DYNAMIC columns + elo_diff (see docs/Column_Analysis.md).
STATIC and semi-dynamic columns are excluded.

In [4]:
feature_cols = [
    'elo_diff',
    'winner_past3', 'loser_past3',
    'winner_head2head_percentage', 'loser_head2head_percentage',
    'winner_head2head_freq', 'loser_head2head_freq',
    'winner_mirage', 'loser_mirage',
    'winner_inferno', 'loser_inferno',
    'winner_nuke', 'loser_nuke',
    'winner_dust2', 'loser_dust2',
    'winner_overpass', 'loser_overpass',
    'winner_train', 'loser_train',
    'winner_ancient', 'loser_ancient',
    'winner_vertigo', 'loser_vertigo',
    'winner_anubis', 'loser_anubis',
    'team1_wins', 'team2_wins', 'team1_losses', 'team2_losses',
    'team1_totalwinrate', 'team2_totalwinrate',
    'team1_totallossrate', 'team2_totallossrate',
    'team1_online_winrate', 'team2_online_winrate',
    'team1_lan_winrate', 'team2_lan_winrate',
    'team1_overall_winrate', 'team2_overall_winrate',
    'event_type',
]

print(f"Total features: {len(feature_cols)}")
missing = [c for c in feature_cols if c not in df.columns]
print("Missing columns (should be empty):", missing)

Total features: 40
Missing columns (should be empty): []


## ⚠️ Important note on winner/loser columns

`winner_past3`, `winner_head2head_*`, `winner_mirage`, etc. are named
relative to the ACTUAL winner of each row — this is a problem, because
at prediction time we don't know the winner yet. These must be
reframed as team1/team2 (not winner/loser) before training, or the
model would be trivially leaking the answer.

In [5]:
def reframe_winner_loser(df, col_base):
    """
    Converts winner_X / loser_X into team1_X / team2_X based on
    whether team1 or team2 was the actual winner in that row.
    """
    winner_col = f'winner_{col_base}'
    loser_col = f'loser_{col_base}'
    team1_col = f'team1_{col_base}'
    team2_col = f'team2_{col_base}'

    is_team1_winner = df['winner'] == 'team1'

    df[team1_col] = np.where(is_team1_winner, df[winner_col], df[loser_col])
    df[team2_col] = np.where(is_team1_winner, df[loser_col], df[winner_col])
    return df

reframe_bases = [
    'past3', 'head2head_percentage', 'head2head_freq',
    'mirage', 'inferno', 'nuke', 'dust2', 'overpass',
    'train', 'ancient', 'vertigo', 'anubis'
]

for base in reframe_bases:
    df = reframe_winner_loser(df, base)

logging.info(f"Reframed {len(reframe_bases)} winner/loser column pairs to team1/team2")
df[['team1_past3', 'team2_past3', 'team1_mirage', 'team2_mirage']].head(3)

,team1_past3,team2_past3,team1_mirage,team2_mirage
0,71.43,80.00,58.4,60.2
1,64.29,55.81,62.0,44.6
2,100.00,50.00,12.5,51.6


In [6]:
final_features = ['elo_diff']

for base in reframe_bases:
    t1c, t2c = f'team1_{base}', f'team2_{base}'
    diff_col = f'{base}_diff'
    df[diff_col] = df[t1c] - df[t2c]
    final_features.append(diff_col)

count_cols = [
    'wins', 'losses', 'totalwinrate', 'totallossrate',
    'online_winrate', 'lan_winrate', 'overall_winrate'
]
for base in count_cols:
    t1c, t2c = f'team1_{base}', f'team2_{base}'
    diff_col = f'{base}_diff'
    df[diff_col] = df[t1c] - df[t2c]
    final_features.append(diff_col)

df['event_type_lan'] = (df['event_type'] == 'lan').astype(int)
final_features.append('event_type_lan')

final_features.remove('totallossrate_diff')

print(f"Final feature count: {len(final_features)}")
print(final_features)

logging.info(f"Final features ({len(final_features)}): {final_features}")

Final feature count: 20
['elo_diff', 'past3_diff', 'head2head_percentage_diff', 'head2head_freq_diff', 'mirage_diff', 'inferno_diff', 'nuke_diff', 'dust2_diff', 'overpass_diff', 'train_diff', 'ancient_diff', 'vertigo_diff', 'anubis_diff', 'wins_diff', 'losses_diff', 'totalwinrate_diff', 'online_winrate_diff', 'lan_winrate_diff', 'overall_winrate_diff', 'event_type_lan']


## 2. Target and temporal train/test split

Target: team1_won (1 if team1 won, 0 otherwise).
Split by date — NOT random — to avoid look-ahead bias.

In [7]:
df['team1_won'] = (df['winner'] == 'team1').astype(int)

split_date = df['date'].quantile(0.8, interpolation='nearest')
train_df = df[df['date'] < split_date].copy()
test_df = df[df['date'] >= split_date].copy()

X_train, y_train = train_df[final_features], train_df['team1_won']
X_test, y_test = test_df[final_features], test_df['team1_won']

print(f"Split date: {split_date}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
logging.info(f"Temporal split at {split_date} | Train: {X_train.shape} | Test: {X_test.shape}")

Split date: 2025-09-04 13:30:00+00:00
Train: (5590, 20), Test: (1399, 20)


## 3. Handle missing values

Some diff features may be NaN if a team has no prior history for
that column. Impute with 0 (neutral — no advantage either way).

In [8]:
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("Remaining NaNs (train):", X_train.isna().sum().sum())
print("Remaining NaNs (test):", X_test.isna().sum().sum())

Remaining NaNs (train): 0
Remaining NaNs (test): 0


## 4. MLflow setup + train Logistic Regression (baseline classifier)

Each run logs: run name/hypothesis + data split identifier,
model parameters + random seed, primary/supporting metrics, and a
confusion matrix artifact — following the "clean run" standard
(Module 8 / Class 4).

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

mlflow.set_experiment("cs2-seeding")

with mlflow.start_run(run_name="logreg_baseline"):
    # --- 1. Run identity: hypothesis + split/data identifier ---
    mlflow.set_tag("hypothesis", "Dynamic features + Elo predict match winner better than Elo alone")
    mlflow.log_param("data_version", "cleaned_matches_v1")
    mlflow.log_param("train_date_range", f"{train_df['date'].min()} to {train_df['date'].max()}")
    mlflow.log_param("test_date_range", f"{test_df['date'].min()} to {test_df['date'].max()}")
    mlflow.log_param("split_date", str(split_date))

    # --- 2. Model/config parameters + random seed ---
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("n_features", len(final_features))
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_seed", 42)

    logreg = LogisticRegression(max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train)

    probs = logreg.predict_proba(X_test_scaled)[:, 1]
    preds = logreg.predict(X_test_scaled)

    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    brier = brier_score_loss(y_test, probs)

    # --- 3. Primary + supporting metrics ---
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("brier_score", brier)

    # --- 4. Artifact: confusion matrix ---
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax, cmap='Blues')
    ax.set_title("Logistic Regression — Confusion Matrix")
    plt.tight_layout()
    plt.savefig('confusion_matrix_logreg.png')
    mlflow.log_artifact('confusion_matrix_logreg.png')
    plt.close()

    mlflow.set_tag("project", "cs2-seeding")
    mlflow.set_tag("stage", "baseline_classifier")

    print(f"LogReg — Accuracy: {acc:.4f}, AUC: {auc:.4f}, Brier: {brier:.4f}")
    logging.info(f"LogReg — Accuracy: {acc:.4f}, AUC: {auc:.4f}, Brier: {brier:.4f}")

2026/08/11 12:47:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/11 12:47:33 INFO mlflow.store.db.utils: Updating database tables
2026/08/11 12:47:35 INFO mlflow.tracking.fluent: Experiment with name 'cs2-seeding' does not exist. Creating a new experiment.


LogReg — Accuracy: 0.7591, AUC: 0.8409, Brier: 0.1612


## 5. Train XGBoost — hyperparameter tuning (main model candidate)

4 parameter combinations compared via TimeSeriesSplit cross-validation
(time-aware, no leakage). Each combination logged as a separate MLflow
run with parameters, random seed, CV + test metrics.

In [10]:
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# XGBoost daraxt-asoslangan model — feature scaling shart emas, shuning uchun
# bu yerda X_train / X_test (scale qilinmagan) ishlatiladi

param_grid = [
    dict(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=300, max_depth=3, learning_rate=0.1,  subsample=0.7, colsample_bytree=0.7),
    dict(n_estimators=500, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8),
]

tscv = TimeSeriesSplit(n_splits=5)  # vaqt bo'yicha to'g'ri CV, tasodifiy emas

best_auc = -1
best_params = None
best_model = None
best_preds = None

for i, params in enumerate(param_grid):
    with mlflow.start_run(run_name=f"xgboost_tune_{i}"):
        # --- 1. Run identity: hypothesis + split/data identifier ---
        mlflow.set_tag("hypothesis", "Tuned XGBoost may outperform Logistic Regression baseline")
        mlflow.log_param("data_version", "cleaned_matches_v1")
        mlflow.log_param("train_date_range", f"{train_df['date'].min()} to {train_df['date'].max()}")
        mlflow.log_param("test_date_range", f"{test_df['date'].min()} to {test_df['date'].max()}")

        # --- 2. Model/config parameters + random seed ---
        mlflow.log_params(params)
        mlflow.log_param("model_type", "XGBoost")
        mlflow.log_param("random_seed", 42)
        mlflow.log_param("cv_splits", 5)

        model = XGBClassifier(**params, random_state=42, eval_metric='logloss')

        cv_scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring='roc_auc')
        cv_mean, cv_std = cv_scores.mean(), cv_scores.std()

        model.fit(X_train, y_train)
        test_probs = model.predict_proba(X_test)[:, 1]
        test_preds = model.predict(X_test)

        acc = accuracy_score(y_test, test_preds)
        auc = roc_auc_score(y_test, test_probs)
        brier = brier_score_loss(y_test, test_probs)

        # --- 3. Primary + supporting metrics ---
        mlflow.log_metric("cv_auc_mean", cv_mean)
        mlflow.log_metric("cv_auc_std", cv_std)
        mlflow.log_metric("test_accuracy", acc)
        mlflow.log_metric("test_auc", auc)
        mlflow.log_metric("test_brier", brier)

        # --- 4. Artifact: confusion matrix ---
        fig, ax = plt.subplots(figsize=(5, 5))
        ConfusionMatrixDisplay.from_predictions(y_test, test_preds, ax=ax, cmap='Oranges')
        ax.set_title(f"XGBoost [{i}] — Confusion Matrix")
        plt.tight_layout()
        cm_path = f'confusion_matrix_xgb_{i}.png'
        plt.savefig(cm_path)
        mlflow.log_artifact(cm_path)
        plt.close()

        mlflow.set_tag("project", "cs2-seeding")
        mlflow.set_tag("stage", "xgboost_tuning")

        print(f"[{i}] {params}")
        print(f"    CV AUC: {cv_mean:.4f} ± {cv_std:.4f} | Test AUC: {auc:.4f} | Acc: {acc:.4f} | Brier: {brier:.4f}")
        logging.info(f"[{i}] CV AUC: {cv_mean:.4f}±{cv_std:.4f} | Test AUC: {auc:.4f} | Acc: {acc:.4f} | Brier: {brier:.4f}")

        if cv_mean > best_auc:
            best_auc = cv_mean
            best_params = params
            best_model = model
            best_preds = test_preds

print("\nBest params:", best_params)
print("Best CV AUC:", best_auc)

xgb = best_model  # keyingi cell'larda ishlatiladigan yakuniy model

[0] {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
    CV AUC: 0.7777 ± 0.0592 | Test AUC: 0.8398 | Acc: 0.7534 | Brier: 0.1617
[1] {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
    CV AUC: 0.7693 ± 0.0614 | Test AUC: 0.8369 | Acc: 0.7534 | Brier: 0.1641
[2] {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.7, 'colsample_bytree': 0.7}
    CV AUC: 0.7600 ± 0.0638 | Test AUC: 0.8390 | Acc: 0.7477 | Brier: 0.1633
[3] {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
    CV AUC: 0.7716 ± 0.0604 | Test AUC: 0.8392 | Acc: 0.7520 | Brier: 0.1622

Best params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}
Best CV AUC: 0.7777472683784283


## 6. Elo-only baseline (for comparison)

Uses elo_diff alone (via a simple logistic transform) as a naive baseline.

In [11]:
elo_probs = 1 / (1 + 10 ** (-X_test['elo_diff'] / 400))
elo_preds = (elo_probs >= 0.5).astype(int)

elo_acc = accuracy_score(y_test, elo_preds)
elo_auc = roc_auc_score(y_test, elo_probs)
elo_brier = brier_score_loss(y_test, elo_probs)

print(f"Elo-only — Accuracy: {elo_acc:.4f}, AUC: {elo_auc:.4f}, Brier: {elo_brier:.4f}")
logging.info(f"Elo-only — Accuracy: {elo_acc:.4f}, AUC: {elo_auc:.4f}, Brier: {elo_brier:.4f}")

Elo-only — Accuracy: 0.6033, AUC: 0.6277, Brier: 0.2368


## 7. Feature importance (Logistic Regression — final model)

Coefficients shown instead of XGBoost's feature_importances_, since
Logistic Regression was selected as the final model (comparable or
better AUC/Accuracy/Brier, simpler and more interpretable).

In [12]:
coef_importance = pd.Series(logreg.coef_[0], index=final_features).sort_values(key=abs, ascending=False)
print("Logistic Regression coefficients (sorted by magnitude):")
print(coef_importance.head(15))

Logistic Regression coefficients (sorted by magnitude):
totalwinrate_diff            1.673623
losses_diff                  0.296209
wins_diff                   -0.232673
elo_diff                     0.153798
online_winrate_diff         -0.121294
nuke_diff                    0.104428
head2head_percentage_diff    0.080238
train_diff                   0.073378
inferno_diff                -0.065427
past3_diff                   0.057241
lan_winrate_diff            -0.053404
dust2_diff                   0.048731
overall_winrate_diff        -0.045098
head2head_freq_diff          0.043983
anubis_diff                  0.027003
dtype: float64


## 8. Save the final model

Logistic Regression selected as the final model — comparable/better
AUC, Accuracy, and Brier score vs. tuned XGBoost, with the added
benefit of simplicity and interpretability (see Section 7).

In [13]:
import pickle

with open('models/logreg_model.pkl', 'wb') as f:
    pickle.dump(logreg, f)

with open('models/feature_list.pkl', 'wb') as f:
    pickle.dump(final_features, f)

logging.info("Saved logreg_model.pkl and feature_list.pkl")
print("Saved.")

Saved.


In [14]:
try:
    from google.colab import files
    import shutil

    files.download('models/logreg_model.pkl')
    files.download('models/feature_list.pkl')
    files.download('data/04_model_training.log')

    if os.path.exists('mlruns'):
        shutil.make_archive('mlruns_backup', 'zip', 'mlruns')
        files.download('mlruns_backup.zip')
    else:
        print("mlruns/ topilmadi — MLflow run qilinmagan bo'lishi mumkin, o'tkazib yuborildi")
except ImportError:
    print("Not running in Colab — files saved locally.")

Not running in Colab — files saved locally.
